# OCR models on Google Colab

Run this notebook from VS Code using your Google Colab extension, or open it directly in Google Colab. Select a Python 3 GPU runtime such as T4/L4/A100; these models use CUDA and will not run on a TPU runtime.

Expected input folder: `/content/drive/MyDrive/tubitak_sample_docs`

Output base folder: `/content/drive/MyDrive/output/output_ocr`

The notebook finds every PDF/image in the input folder and runs both OCR models over all files. Each model writes one text-only Markdown file in its own output folder:

- `paddleocr_vl_1_6/paddleocr_vl_1_6_output.md`
- `deepseek_ocr_2/deepseek_ocr_2_output.md`

In [ ]:
# Check that the Colab runtime has an NVIDIA GPU such as T4/L4/A100.
!nvidia-smi || true

In [ ]:
# Install runtime dependencies. Run this once per fresh Colab runtime.
!apt-get update -qq
!apt-get install -y -qq poppler-utils

# PaddleOCR-VL needs PaddlePaddle. The GPU wheel is preferred on Colab GPU runtimes;
# the CPU package fallback keeps the install cell from blocking if the GPU wheel index changes.
!python -m pip install -q -U paddlepaddle-gpu --extra-index-url https://www.paddlepaddle.org.cn/packages/stable/cu126/ || python -m pip install -q -U paddlepaddle
%pip install -q -U paddleocr

# DeepSeek-OCR-2 and PDF/image utilities.
%pip install -q "transformers==4.46.3" "tokenizers==0.20.3" torch torchvision accelerate einops addict easydict safetensors timm pillow pdf2image

# DeepSeek recommends flash-attn for GPU inference. This can fail on some Colab images,
# so the DeepSeek cell below falls back if flash attention is unavailable.
!pip install -q flash-attn==2.7.3 --no-build-isolation || true

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import re
import shutil

INPUT_DIR = Path('/content/drive/MyDrive/tubitak_sample_docs')
OUTPUT_BASE_DIR = Path('/content/drive/MyDrive/output/output_ocr')
PADDLE_VL_OUTPUT_DIR = OUTPUT_BASE_DIR / 'paddleocr_vl_1_6'
DEEPSEEK_OUTPUT_DIR = OUTPUT_BASE_DIR / 'deepseek_ocr_2'
PADDLE_VL_OUTPUT_FILE = PADDLE_VL_OUTPUT_DIR / 'paddleocr_vl_1_6_output.md'
DEEPSEEK_OUTPUT_FILE = DEEPSEEK_OUTPUT_DIR / 'deepseek_ocr_2_output.md'

SUPPORTED_EXTENSIONS = {'.pdf', '.png', '.jpg', '.jpeg', '.tif', '.tiff', '.bmp', '.webp'}

for output_dir in [PADDLE_VL_OUTPUT_DIR, DEEPSEEK_OUTPUT_DIR]:
    output_dir.mkdir(parents=True, exist_ok=True)
    for existing_path in output_dir.iterdir():
        if existing_path.is_file() or existing_path.is_symlink():
            existing_path.unlink()
        elif existing_path.is_dir():
            shutil.rmtree(existing_path)

if not INPUT_DIR.exists():
    raise FileNotFoundError(f'Input directory does not exist: {INPUT_DIR}')

INPUT_FILES = sorted(
    [path for path in INPUT_DIR.rglob('*') if path.is_file() and path.suffix.lower() in SUPPORTED_EXTENSIONS],
    key=lambda path: path.relative_to(INPUT_DIR).as_posix().lower(),
)

if not INPUT_FILES:
    raise FileNotFoundError(f'No supported PDF/image files found under: {INPUT_DIR}')

IMAGE_MARKDOWN_RE = re.compile(r'!\[[^\]]*\]\([^)]*\)')
HTML_IMAGE_RE = re.compile(r'<img\b[^>]*>', flags=re.IGNORECASE)

def strip_image_references(text):
    text = IMAGE_MARKDOWN_RE.sub('', text)
    text = HTML_IMAGE_RE.sub('', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

def read_text_artifacts(root_dir):
    root_dir = Path(root_dir)
    chunks = []
    for path in sorted(root_dir.rglob('*')):
        if path.is_file() and path.suffix.lower() in {'.md', '.markdown', '.txt'}:
            chunks.append(strip_image_references(path.read_text(encoding='utf-8', errors='ignore')))
    return '\n\n'.join(chunk for chunk in chunks if chunk)

def file_header(path):
    relative_path = path.relative_to(INPUT_DIR).as_posix()
    return f'\n\n====================\nFILE: {relative_path}\n====================\n'

def page_header(page_number):
    return f'\n--- Page {page_number} ---\n'

def safe_stem(path):
    return re.sub(r'[^A-Za-z0-9_.-]+', '_', path.stem).strip('._') or 'document'

print(f'Input directory: {INPUT_DIR}')
print(f'Output base directory: {OUTPUT_BASE_DIR}')
print(f'PaddleOCR-VL output directory: {PADDLE_VL_OUTPUT_DIR}')
print(f'DeepSeek output directory: {DEEPSEEK_OUTPUT_DIR}')
print(f'Found {len(INPUT_FILES)} supported input files:')
for input_file in INPUT_FILES:
    print(f' - {input_file.relative_to(INPUT_DIR).as_posix()}')

## Run PaddleOCR-VL-1.6

This cell reuses the existing PaddleOCR-VL pipeline when rerun in the same Colab runtime. It saves only the combined text/Markdown string output, not JSON files, rendered images, or asset ZIPs.

In [ ]:
import tempfile
import time
from datetime import datetime
from paddleocr import PaddleOCRVL

start_time = time.perf_counter()
print(f'[DEBUG] Started PaddleOCR-VL at {datetime.now().isoformat(timespec="seconds")}')
print(f'[DEBUG] Output directory: {PADDLE_VL_OUTPUT_DIR}')

if 'paddle_vl_pipeline' not in globals():
    paddle_vl_pipeline = PaddleOCRVL(batch_size=1, pipeline_version='v1.6')
    print('[DEBUG] Initialized PaddleOCR-VL pipeline')
else:
    print('[DEBUG] Reusing existing PaddleOCR-VL pipeline to avoid PDX reinitialization')

output_chunks = ['# PaddleOCR-VL-1.6 OCR Output\n']

for file_index, input_path in enumerate(INPUT_FILES, start=1):
    file_start_time = time.perf_counter()
    print(f'[DEBUG] PaddleOCR-VL processing {file_index}/{len(INPUT_FILES)}: {input_path.name}')
    output_chunks.append(file_header(input_path))

    try:
        with tempfile.TemporaryDirectory() as staging_dir:
            result_count = 0
            for result_count, res in enumerate(paddle_vl_pipeline.predict(str(input_path)), start=1):
                res.save_to_markdown(save_path=staging_dir)

            text = read_text_artifacts(staging_dir)
            if not text:
                text = '(No text returned.)'

        output_chunks.append(text.strip())
        print(
            f'[DEBUG] PaddleOCR-VL finished {input_path.name} with {result_count} result(s) '
            f'in {time.perf_counter() - file_start_time:.2f}s'
        )
    except Exception as exc:
        error_text = f'[ERROR] {type(exc).__name__}: {exc}'
        output_chunks.append(error_text)
        print(f'[DEBUG] PaddleOCR-VL failed for {input_path.name}: {error_text}')

PADDLE_VL_OUTPUT_FILE.write_text('\n'.join(output_chunks).strip() + '\n', encoding='utf-8')

print(f'[DEBUG] Saved PaddleOCR-VL text output to: {PADDLE_VL_OUTPUT_FILE}')

print(f'[DEBUG] Finished PaddleOCR-VL at {datetime.now().isoformat(timespec="seconds")}')
print(f'[DEBUG] Total elapsed time: {time.perf_counter() - start_time:.2f}s')

## Run DeepSeek-OCR-2

DeepSeek-OCR-2 is much heavier. This cell converts each PDF page to a temporary PNG before inference, processes all files one by one to reduce memory pressure on T4, and saves only the combined text/Markdown string output.

In [ ]:
import gc
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True,max_split_size_mb:64')

import tempfile
import time
from datetime import datetime

import torch
from pdf2image import convert_from_path, pdfinfo_from_path
from PIL import Image, ImageOps
from transformers import AutoModel, AutoTokenizer

if not torch.cuda.is_available():
    raise RuntimeError('DeepSeek-OCR-2 needs a GPU runtime. In Colab, select Runtime > Change runtime type > GPU.')

DEEPSEEK_MODEL_NAME = 'deepseek-ai/DeepSeek-OCR-2'
DEEPSEEK_PROMPT = '<image>\n<|grounding|>Convert the document to markdown. '

# Low-VRAM defaults for Colab T4. Raise these later only if the OCR quality is not enough.
DEEPSEEK_LOW_VRAM = True
if DEEPSEEK_LOW_VRAM:
    DEEPSEEK_PDF_DPI = 150
    DEEPSEEK_BASE_SIZE = 768
    DEEPSEEK_IMAGE_SIZE = 512
    DEEPSEEK_MAX_IMAGE_SIDE = 1600
    DEEPSEEK_CROP_MODE = False
else:
    DEEPSEEK_PDF_DPI = 200
    DEEPSEEK_BASE_SIZE = 1024
    DEEPSEEK_IMAGE_SIZE = 768
    DEEPSEEK_MAX_IMAGE_SIDE = 2200
    DEEPSEEK_CROP_MODE = True

# Keep this False for text-only output and lower memory use.
DEEPSEEK_SAVE_RESULTS = False
DEEPSEEK_RETRY_BASE_SIZE = 512
DEEPSEEK_RETRY_IMAGE_SIZE = 384
DEEPSEEK_RESAMPLE_LANCZOS = getattr(getattr(Image, 'Resampling', Image), 'LANCZOS')

start_time = time.perf_counter()
print(f'[DEBUG] Started DeepSeek OCR at {datetime.now().isoformat(timespec="seconds")}')
print(f'[DEBUG] Output directory: {DEEPSEEK_OUTPUT_DIR}')
print(f'[DEBUG] GPU: {torch.cuda.get_device_name(0)}')
print(
    '[DEBUG] DeepSeek low-VRAM settings: '
    f'dpi={DEEPSEEK_PDF_DPI}, base_size={DEEPSEEK_BASE_SIZE}, '
    f'image_size={DEEPSEEK_IMAGE_SIZE}, max_image_side={DEEPSEEK_MAX_IMAGE_SIDE}, '
    f'crop_mode={DEEPSEEK_CROP_MODE}, save_results={DEEPSEEK_SAVE_RESULTS}'
)

def log_cuda_memory(label):
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    print(f'[DEBUG] CUDA memory {label}: allocated={allocated:.2f} GiB, reserved={reserved:.2f} GiB')

if 'deepseek_tokenizer' not in globals():
    tokenizer_start_time = time.perf_counter()
    deepseek_tokenizer = AutoTokenizer.from_pretrained(DEEPSEEK_MODEL_NAME, trust_remote_code=True)
    print(f'[DEBUG] Tokenizer loaded in {time.perf_counter() - tokenizer_start_time:.2f}s')
else:
    print('[DEBUG] Reusing existing DeepSeek tokenizer')

deepseek_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(f'[DEBUG] DeepSeek dtype: {deepseek_dtype}')

if 'deepseek_model' not in globals():
    model_start_time = time.perf_counter()
    try:
        deepseek_model = AutoModel.from_pretrained(
            DEEPSEEK_MODEL_NAME,
            _attn_implementation='flash_attention_2',
            trust_remote_code=True,
            use_safetensors=True,
            low_cpu_mem_usage=True,
            torch_dtype=deepseek_dtype,
        )
    except Exception as exc:
        print(f'[DEBUG] Flash attention load failed; falling back. Reason: {exc}')
        deepseek_model = AutoModel.from_pretrained(
            DEEPSEEK_MODEL_NAME,
            trust_remote_code=True,
            use_safetensors=True,
            low_cpu_mem_usage=True,
            torch_dtype=deepseek_dtype,
        )
    deepseek_model = deepseek_model.eval().requires_grad_(False).cuda().to(deepseek_dtype)
    torch.cuda.empty_cache()
    log_cuda_memory('after model load')
    print(f'[DEBUG] Model loaded in {time.perf_counter() - model_start_time:.2f}s')
else:
    deepseek_model = deepseek_model.eval().requires_grad_(False).cuda().to(deepseek_dtype)
    torch.cuda.empty_cache()
    log_cuda_memory('after model reuse')
    print('[DEBUG] Reusing existing DeepSeek model')

def save_resized_rgb(image, image_path):
    image = ImageOps.exif_transpose(image).convert('RGB')
    max_side = max(image.size)
    if max_side > DEEPSEEK_MAX_IMAGE_SIDE:
        scale = DEEPSEEK_MAX_IMAGE_SIDE / max_side
        new_size = (max(1, int(image.width * scale)), max(1, int(image.height * scale)))
        image = image.resize(new_size, DEEPSEEK_RESAMPLE_LANCZOS)
    image.save(image_path, optimize=True)

def deepseek_image_inputs(input_path, work_dir):
    input_path = Path(input_path)
    work_dir = Path(work_dir)
    if input_path.suffix.lower() == '.pdf':
        page_count = int(pdfinfo_from_path(str(input_path)).get('Pages', 0))
        for page_number in range(1, page_count + 1):
            page_image = convert_from_path(
                str(input_path),
                dpi=DEEPSEEK_PDF_DPI,
                first_page=page_number,
                last_page=page_number,
            )[0]
            image_path = work_dir / f'{safe_stem(input_path)}_page_{page_number:04d}.png'
            save_resized_rgb(page_image, image_path)
            page_image.close()
            yield page_number, image_path
    else:
        image_path = work_dir / f'{safe_stem(input_path)}_image.png'
        with Image.open(input_path) as image:
            save_resized_rgb(image, image_path)
        yield None, image_path

def run_deepseek_image(image_path, base_size, image_size, crop_mode):
    with tempfile.TemporaryDirectory() as staging_dir:
        with torch.inference_mode(), torch.autocast(device_type='cuda', dtype=deepseek_dtype):
            result = deepseek_model.infer(
                deepseek_tokenizer,
                prompt=DEEPSEEK_PROMPT,
                image_file=str(image_path),
                output_path=staging_dir,
                base_size=base_size,
                image_size=image_size,
                crop_mode=crop_mode,
                save_results=DEEPSEEK_SAVE_RESULTS,
            )
        if result is not None and str(result).strip():
            return strip_image_references(str(result))
        text = read_text_artifacts(staging_dir)
        return text if text else '(No text returned.)'

infer_start_time = time.perf_counter()
DEEPSEEK_OUTPUT_FILE.write_text('# DeepSeek-OCR-2 OCR Output\n', encoding='utf-8')

def append_deepseek_output(text):
    with DEEPSEEK_OUTPUT_FILE.open('a', encoding='utf-8') as output_stream:
        output_stream.write(text)

for file_index, input_path in enumerate(INPUT_FILES, start=1):
    file_start_time = time.perf_counter()
    print(f'[DEBUG] DeepSeek processing {file_index}/{len(INPUT_FILES)}: {input_path.name}')
    append_deepseek_output(file_header(input_path))

    try:
        with tempfile.TemporaryDirectory() as conversion_dir:
            for page_number, image_path in deepseek_image_inputs(input_path, conversion_dir):
                page_start_time = time.perf_counter()
                if page_number is not None:
                    append_deepseek_output(page_header(page_number))

                page_label = f' page {page_number}' if page_number is not None else ''
                try:
                    text = run_deepseek_image(
                        image_path,
                        base_size=DEEPSEEK_BASE_SIZE,
                        image_size=DEEPSEEK_IMAGE_SIZE,
                        crop_mode=DEEPSEEK_CROP_MODE,
                    )
                except torch.cuda.OutOfMemoryError:
                    print(f'[DEBUG] CUDA OOM on {input_path.name}{page_label}; retrying with emergency smaller settings')
                    gc.collect()
                    torch.cuda.empty_cache()
                    text = run_deepseek_image(
                        image_path,
                        base_size=DEEPSEEK_RETRY_BASE_SIZE,
                        image_size=DEEPSEEK_RETRY_IMAGE_SIZE,
                        crop_mode=False,
                    )

                append_deepseek_output(text.strip() + '\n')
                print(
                    f'[DEBUG] DeepSeek finished {input_path.name}'
                    f'{page_label} '
                    f'in {time.perf_counter() - page_start_time:.2f}s'
                )
                gc.collect()
                torch.cuda.empty_cache()
                log_cuda_memory(f'after {input_path.name}{page_label}')

        print(f'[DEBUG] DeepSeek file completed in {time.perf_counter() - file_start_time:.2f}s: {input_path.name}')
    except Exception as exc:
        error_text = f'[ERROR] {type(exc).__name__}: {exc}'
        append_deepseek_output(error_text + '\n')
        print(f'[DEBUG] DeepSeek failed for {input_path.name}: {error_text}')

print(f'[DEBUG] Saved DeepSeek text output to: {DEEPSEEK_OUTPUT_FILE}')

print(f'[DEBUG] Inference completed in {time.perf_counter() - infer_start_time:.2f}s')
print(f'[DEBUG] Finished DeepSeek OCR at {datetime.now().isoformat(timespec="seconds")}')
print(f'[DEBUG] Total elapsed time: {time.perf_counter() - start_time:.2f}s')